In [ ]:
ENABLE_TUNING = True
MAX_TUNING_TRIALS = 5
TUNING_EPOCHS = 5

print(f"Tuning: {'AÇIK' if ENABLE_TUNING else 'KAPALI'}")
print(f"Max deneme: {MAX_TUNING_TRIALS}")
print(f"Deneme başına epoch: {TUNING_EPOCHS}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q kaggle keras-tuner
print("Paketler kuruldu!")

In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = 'kullanici_adiniz'
os.environ['KAGGLE_KEY'] = 'api_keyiniz'

os.makedirs('/root/.kaggle', exist_ok=True)
if os.path.exists('/content/kaggle.json'):
    !cp /content/kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json
    print("Kaggle API key ayarlandı (dosyadan)!")
else:
    print("Kaggle API key ayarlandı (environment'dan)!")

In [ ]:
import shutil
from pathlib import Path
import random

TARGET_CLASSES = [
    'cardboard', 'food_organics', 'glass', 'metal',
    'miscellaneous_trash', 'paper', 'plastic', 'textile_trash', 'vegetation'
]

CLASS_MAPPINGS = {
    'cardboard': 'cardboard', 'glass': 'glass', 'metal': 'metal',
    'paper': 'paper', 'plastic': 'plastic', 'trash': 'miscellaneous_trash',
    'battery': 'miscellaneous_trash', 'biological': 'food_organics',
    'brown-glass': 'glass', 'green-glass': 'glass', 'white-glass': 'glass',
    'clothes': 'textile_trash', 'shoes': 'textile_trash',
    'food_organics': 'food_organics', 'miscellaneous_trash': 'miscellaneous_trash',
    'textile_trash': 'textile_trash', 'vegetation': 'vegetation',
}

MERGED_DIR = Path('/content/merged_dataset')
for cls in TARGET_CLASSES:
    (MERGED_DIR / cls).mkdir(parents=True, exist_ok=True)

def copy_images(src_dir, prefix):
    count = 0
    src_path = Path(src_dir)
    if not src_path.exists():
        return 0
    for class_dir in src_path.iterdir():
        if not class_dir.is_dir():
            continue
        src_class = class_dir.name.lower().replace(' ', '_')
        dst_class = CLASS_MAPPINGS.get(src_class)
        if not dst_class:
            continue
        for img in class_dir.glob('*'):
            if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                dst_name = f"{prefix}_{src_class}_{img.name}"
                dst_file = MERGED_DIR / dst_class / dst_name
                if not dst_file.exists():
                    shutil.copy(img, dst_file)
                    count += 1
    return count

print("1. RealWaste isleniyor...")
for p in [Path(DATASETS_DIR) / 'RealWaste', Path(DATASETS_DIR) / 'realwaste', Path(DATASETS_DIR)]:
    c = copy_images(p, 'rw')
    if c > 0:
        print(f"   {c} goruntu eklendi")
        break

print("2. Garbage Classification isleniyor...")
gc_total = 0
for p in [Path(DATASETS_DIR) / 'DATASET' / 'TRAIN', Path(DATASETS_DIR) / 'DATASET' / 'TEST']:
    gc_total += copy_images(p, 'gc')
print(f"   {gc_total} goruntu eklendi")

print("3. TrashNet isleniyor...")
for p in [Path(DATASETS_DIR) / 'Garbage classification' / 'Garbage classification']:
    c = copy_images(p, 'tn')
    if c > 0:
        print(f"   {c} goruntu eklendi")
        break

print("\n" + "="*50)
print("BIRLESTIRILMIS VERI SETI")
print("="*50)
total = 0
for cls in TARGET_CLASSES:
    count = len(list((MERGED_DIR / cls).glob('*')))
    total += count
    print(f"{cls:25} : {count:5}")
print("-"*50)
print(f"{'TOPLAM':25} : {total:5}")

In [ ]:
FINAL_DIR = Path('/content/waste_dataset')
random.seed(42)

for split in ['train', 'val', 'test']:
    for cls in TARGET_CLASSES:
        (FINAL_DIR / split / cls).mkdir(parents=True, exist_ok=True)

print("Veri seti bolunuyor...\n")
split_stats = {'train': 0, 'val': 0, 'test': 0}

for cls in TARGET_CLASSES:
    images = list((MERGED_DIR / cls).glob('*'))
    random.shuffle(images)
    n = len(images)
    n_train, n_val = int(n * 0.7), int(n * 0.15)
    
    for split, imgs in [('train', images[:n_train]), ('val', images[n_train:n_train+n_val]), ('test', images[n_train+n_val:])]:
        for img in imgs:
            shutil.copy(img, FINAL_DIR / split / cls / img.name)
        split_stats[split] += len(imgs)
    print(f"{cls}: train={n_train}, val={n_val}, test={n-n_train-n_val}")

print(f"\nToplam: train={split_stats['train']}, val={split_stats['val']}, test={split_stats['test']}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_true, y_pred = [], []
for images, labels in test_ds_prep:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CONFIG['classes'], yticklabels=CONFIG['classes'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Tahmin')
plt.ylabel('Gercek')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/confusion_matrix.png', dpi=300)
plt.show()

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
report = classification_report(y_true, y_pred, target_names=CONFIG['classes'], digits=4)
print(report)

with open(f'{MODEL_SAVE_DIR}/classification_report.txt', 'w') as f:
    f.write(report)